# NSE SQLite Database Analyzer

This interactive notebook uses Jupyter widgets (`ipywidgets`) to query and inspect Capital Market, Derivatives Market, Stock Price History, Corporate Announcements, and Option Chain SQLite databases. 

Use the **Search Table** text box to instantly filter the list of tables (highly useful for navigating 1,600+ stocks in the `stock_data.db` database!).

In [1]:
# Install dependencies if not already installed
# %pip install ipywidgets pandas --upgrade

In [1]:
import os
import sqlite3
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from config import (
    CAPITAL_MARKET_DB,
    DERIVATIVES_MARKET_DB,
    STOCK_DATA_DB,
    CORPORATE_DB,
    OPTION_CHAIN_DB
)

# 1. Fetch database summary details
def get_database_summary(db_name):
    if not os.path.exists(db_name):
        return {
            "exists": False,
            "file_size_kb": 0,
            "table_count": 0,
            "table_details": pd.DataFrame()
        }
        
    try:
        file_size_kb = os.path.getsize(db_name) / 1024
        conn = sqlite3.connect(db_name)
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tables = sorted([row[0] for row in cursor.fetchall()])
        
        table_details = []
        for t in tables:
            try:
                cursor.execute(f"SELECT COUNT(*) FROM {t}")
                row_count = cursor.fetchone()[0]
                table_details.append({"Table Name": t, "Row Count": row_count})
            except Exception:
                table_details.append({"Table Name": t, "Row Count": "Error"})
                
        conn.close()
        return {
            "exists": True,
            "file_size_kb": file_size_kb,
            "table_count": len(tables),
            "table_details": pd.DataFrame(table_details)
        }
    except Exception as e:
        return {
            "exists": False,
            "file_size_kb": 0,
            "table_count": 0,
            "table_details": pd.DataFrame(),
            "error": str(e)
        }

def get_database_tables(db_name):
    summary = get_database_summary(db_name)
    if summary["exists"] and not summary["table_details"].empty:
        return summary["table_details"]["Table Name"].tolist()
    return []

# 2. Define widgets using imported config db values
db_dropdown = widgets.Dropdown(
    options=[
        (f"Capital Market ({CAPITAL_MARKET_DB})", CAPITAL_MARKET_DB),
        (f"Derivatives Market ({DERIVATIVES_MARKET_DB})", DERIVATIVES_MARKET_DB),
        (f"Stock Price History ({STOCK_DATA_DB})", STOCK_DATA_DB),
        (f"Corporate Announcements ({CORPORATE_DB})", CORPORATE_DB),
        (f"Option Chain ({OPTION_CHAIN_DB})", OPTION_CHAIN_DB)
    ],
    value=CAPITAL_MARKET_DB,
    description='Database:',
    disabled=False,
    style={'description_width': 'initial'}
)

search_box = widgets.Text(
    value='',
    placeholder='Type to filter tables (e.g. daily)...',
    description='Filter Tables:',
    disabled=False,
    style={'description_width': 'initial'}
)

table_dropdown = widgets.Dropdown(
    options=get_database_tables(CAPITAL_MARKET_DB),
    description='Table:',
    disabled=False,
    style={'description_width': 'initial'}
)

# NEW: Explicit search box to filter content inside the tables by asset/symbol
symbol_search_box = widgets.Text(
    value='',
    placeholder='Type asset symbol (e.g. sbin, nifty)...',
    description='Symbol Search:',
    disabled=False,
    style={'description_width': 'initial'}
)

use_case_dropdown = widgets.Dropdown(
    options=[
        ("Show Database Summary (File Size & Tables List)", "db_summary"),
        ("Show Update Summary (Timestamps & Counts)", "summary"),
        ("Show Latest Raw Records", "latest"),
        ("Show Top 5 by Volume Surge / Metrics", "top_vol"),
        ("Show Top 5 by Price Change %", "top_price"),
        ("Show Symbol Count Summary (e.g. for Corporate Announcements)", "symbol_summary")
    ],
    value="db_summary",
    description='Use Case:',
    disabled=False,
    style={'description_width': 'initial'}
)

output_area = widgets.Output()

# 3. Change handlers
def filter_tables():
    db_name = db_dropdown.value
    search_query = search_box.value.strip().lower()
    all_tables = get_database_tables(db_name)
    
    if search_query:
        filtered = [t for t in all_tables if search_query in t.lower()]
    else:
        filtered = all_tables
        
    table_dropdown.options = filtered
    if filtered:
        table_dropdown.value = filtered[0]
    else:
        table_dropdown.value = None
    run_query()

def on_db_change(change):
    search_box.value = ''
    symbol_search_box.value = ''  # Reset symbol search on DB swap
    filter_tables()

def on_search_change(change):
    filter_tables()

def on_symbol_change(change):
    run_query()

# Helper to automatically discover and construct SQL WHERE clauses for symbols
def get_symbol_filter_clause(df_columns, search_term):
    if not search_term:
        return "", []
    
    # Identify typical symbol/identifier column flags
    sym_col = next((c for c in df_columns if any(k in c.lower() for k in ['symbol', 'stock', 'index', 'identifier', 'key'])), None)
    if sym_col:
        # Use localized case-insensitive SQL matching
        return f" WHERE LOWER({sym_col}) LIKE ? ", [f"%{search_term.lower()}%"]
    return "", []

# 4. Query execution logic
def run_query(change=None):
    db_name = db_dropdown.value
    table = table_dropdown.value
    use_case = use_case_dropdown.value
    sym_query = symbol_search_box.value.strip()
    
    with output_area:
        clear_output()
        
        if use_case == "db_summary":
            summary = get_database_summary(db_name)
            print(f"========================================================================")
            print(f" DATABASE SUMMARY: {db_name}")
            print(f"========================================================================")
            if summary["exists"]:
                print(f"* File Size: {summary['file_size_kb']:.2f} KB")
                print(f"* Number of Tables: {summary['table_count']}")
                if not summary["table_details"].empty:
                    print("\n* Table Details:")
                    display(summary["table_details"])
            else:
                print("* Database file does not exist on disk or has errors.")
                if "error" in summary:
                    print(f"  Error details: {summary['error']}")
            print(f"========================================================================\n")
            return
            
        if not table:
            print(f"No tables found in database '{db_name}'.")
            return
            
        conn = sqlite3.connect(db_name)
        
        try:
            # Sample structure quickly to extract valid columns for structural mapping
            sample_df = pd.read_sql_query(f"SELECT * FROM {table} LIMIT 1", conn)
            where_clause, params = get_symbol_filter_clause(sample_df.columns, sym_query)
            
            if use_case == "summary":
                if where_clause:
                    query = f"SELECT fetchedAt, COUNT(*) AS [Record Count] FROM {table} {where_clause} GROUP BY fetchedAt ORDER BY fetchedAt DESC"
                    df = pd.read_sql_query(query, conn, params=params)
                else:
                    query = f"SELECT fetchedAt, COUNT(*) AS [Record Count] FROM {table} GROUP BY fetchedAt ORDER BY fetchedAt DESC"
                    df = pd.read_sql_query(query, conn)
                    
                print(f"=== Update Summary for Table: '{table}' (DB: {db_name}) ===\n")
                if sym_query:
                    print(f"Filtering records matching token: '{sym_query}'\n")
                display(df)
                
            elif use_case == "latest":
                if where_clause:
                    query = f"SELECT * FROM {table} {where_clause} AND fetchedAt = (SELECT MAX(fetchedAt) FROM {table})"
                    df = pd.read_sql_query(query, conn, params=params)
                else:
                    query = f"SELECT * FROM {table} WHERE fetchedAt = (SELECT MAX(fetchedAt) FROM {table})"
                    df = pd.read_sql_query(query, conn)
                    
                if not df.empty:
                    print(f"=== Latest Records for Table: '{table}' (DB: {db_name}, Timestamp: {df['fetchedAt'].iloc[0]}) ===\n")
                    display(df)
                else:
                    print(f"No records found matching your selection parameters.")
                    
            elif use_case == "top_vol":
                if where_clause:
                    query = f"SELECT * FROM {table} {where_clause} AND fetchedAt = (SELECT MAX(fetchedAt) FROM {table})"
                    df = pd.read_sql_query(query, conn, params=params)
                else:
                    query = f"SELECT * FROM {table} WHERE fetchedAt = (SELECT MAX(fetchedAt) FROM {table})"
                    df = pd.read_sql_query(query, conn)
                    
                if not df.empty:
                    vol_col = next((c for c in df.columns if any(k in c for k in ['volChange', 'Volume', 'volume', 'Quantity', 'quantity', 'TotalTradedQty', 'totalTradedVolume', 'ce_totaltradedvolume', 'pe_totaltradedvolume'])), None)
                    if vol_col:
                        df[vol_col] = pd.to_numeric(df[vol_col], errors='coerce')
                        top_df = df.sort_values(by=vol_col, ascending=False).head(5)
                        print(f"=== Top 5 Records sorted by Vol/Qty Column '{vol_col}' ===\n")
                        
                        sym_col = next((c for c in df.columns if any(k in c.lower() for k in ['symbol', 'stock', 'index', 'key'])), None)
                        name_col = next((c for c in df.columns if any(k in c.lower() for k in ['name', 'company'])), None)
                        ltp_col = next((c for c in df.columns if any(k in c.lower() for k in ['ltp', 'price', 'last', 'close', 'lastPrice', 'ce_lastprice', 'pe_lastprice'])), None)
                        
                        show_cols = [c for c in [sym_col, name_col, vol_col, ltp_col] if c]
                        display(top_df[show_cols])
                    else:
                        print(f"No volume metrics detected. Displaying up to top 5 filtered entries:")
                        display(df.head(5))
                else:
                    print(f"No records found matching your selection parameters.")
                    
            elif use_case == "top_price":
                if where_clause:
                    query = f"SELECT * FROM {table} {where_clause} AND fetchedAt = (SELECT MAX(fetchedAt) FROM {table})"
                    df = pd.read_sql_query(query, conn, params=params)
                else:
                    query = f"SELECT * FROM {table} WHERE fetchedAt = (SELECT MAX(fetchedAt) FROM {table})"
                    df = pd.read_sql_query(query, conn)
                    
                if not df.empty:
                    price_col = next((c for c in df.columns if any(k in c for k in ['pChange', 'change', 'percent', 'Change', 'DlyQttoTradedQty', 'pchange', 'ce_pchange', 'pe_pchange'])), None)
                    if price_col:
                        df[price_col] = pd.to_numeric(df[price_col], errors='coerce')
                        top_df = df.sort_values(by=price_col, ascending=False).head(5)
                        print(f"=== Top 5 Records sorted by Variance Metric '{price_col}' ===\n")
                        
                        sym_col = next((c for c in df.columns if any(k in c.lower() for k in ['symbol', 'stock', 'index'])), None)
                        ltp_col = next((c for c in df.columns if any(k in c.lower() for k in ['ltp', 'price', 'last', 'close', 'lastPrice', 'ce_lastprice', 'pe_lastprice'])), None)
                        
                        show_cols = [c for c in [sym_col, price_col, ltp_col] if c]
                        display(top_df[show_cols])
                    else:
                        print(f"No pricing delta metrics detected. Displaying up to top 5 filtered entries:")
                        display(df.head(5))
                else:
                    print(f"No records found matching your selection parameters.")
            elif use_case == "symbol_summary":
                sym_col = next((c for c in sample_df.columns if any(k in c.lower() for k in ['symbol', 'stock', 'index', 'key'])), None)
                if sym_col:
                    if where_clause:
                        query = f"SELECT {sym_col}, COUNT(*) AS [Record Count] FROM {table} {where_clause} GROUP BY {sym_col} ORDER BY [Record Count] DESC"
                        res_df = pd.read_sql_query(query, conn, params=params)
                    else:
                        query = f"SELECT {sym_col}, COUNT(*) AS [Record Count] FROM {table} GROUP BY {sym_col} ORDER BY [Record Count] DESC"
                        res_df = pd.read_sql_query(query, conn)
                    
                    print(f"=== Symbol Count Summary for Table: '{table}' (DB: {db_name}) ===\n")
                    if sym_query:
                        print(f"Filtering records matching symbol: '{sym_query}'\n")
                    display(res_df)
                else:
                    print("No symbol column found in this table to group by.")
        except Exception as e:
            print(f"Error querying table '{table}': {e}")
        finally:
            conn.close()

# 5. Bind events to layout changes
db_dropdown.observe(on_db_change, names='value')
search_box.observe(on_search_change, names='value')
symbol_search_box.observe(on_symbol_change, names='value')
table_dropdown.observe(run_query, names='value')
use_case_dropdown.observe(run_query, names='value')

# 6. Render Dashboard Elements
input_layout = widgets.VBox([
    widgets.HBox([db_dropdown, search_box, symbol_search_box]),
    widgets.HBox([table_dropdown, use_case_dropdown])
])
display(input_layout, output_area)

# Initialize
run_query()

Output()